# Epsilon Fund — Momentum Master Portfolio

Combined OOS portfolio that pulls from **any subset of strategies** in `topics/momentum/strategies/`.
Each coin can be sourced from a different strategy, and the same coin can appear from multiple strategies (as separate sleeves).

Individual signals and trade logic are **never modified** — behaviour depends entirely on each strategy's `*_oos.pkl`.

Designed to scale: drop a new folder in `STRATEGIES`, add entries to `SELECTION`, rerun.

---

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import io, contextlib

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows

STRAT_ROOT = os.path.join(ROOT, 'topics', 'momentum', 'strategies')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos
from wf_engine import cost_stress_test

client = get_binance_client()

---
## Strategies Registry
Declare every strategy folder you want available. Add a new entry here whenever a new walk-forward strategy is ready — everything downstream is keyed off this dict.

Each strategy folder must contain `*_oos.pkl` files produced by the walk-forward engine.

In [ ]:
# ── strategy registry ────────────────────────────────────────────────────────
# key = short strategy tag used in SELECTION below
# val = folder name under topics/momentum/strategies/
STRATEGIES = {
    'wf2': 'wf_testing_2',      # momentum #2 (base WF)
    'bb':  'bb_breakout_wf',    # BB breakout walk-forward
    # 'xxx': 'new_strategy_folder',   # ← drop new strategies in here
}

# ── load every strategy's OOS pkls into a nested dict ────────────────────────
strategy_oos = {}   # { strat_tag : { coin : oos_df } }
for tag, folder in STRATEGIES.items():
    path = os.path.join(STRAT_ROOT, folder)
    if not os.path.isdir(path):
        print(f'  [skip] {tag}: folder not found — {path}')
        continue
    d = {}
    for fname in os.listdir(path):
        if fname.endswith('_oos.pkl'):
            coin = fname.replace('_oos.pkl', '').upper()
            d[coin] = pd.read_pickle(os.path.join(path, fname))
    strategy_oos[tag] = d
    print(f'  {tag:<6} ({folder}): {sorted(d.keys())}')

print()
print('Available sleeves (strategy → coin):')
for tag, d in strategy_oos.items():
    for c in sorted(d):
        print(f'  {tag}:{c}')

---
## Sleeve Selection
Pick which **(strategy, coin)** pairs to include in the combined portfolio.

Each entry becomes one **sleeve** in the portfolio. The label is what appears in plots and stat tables — prefer `{COIN}_{strat}` so collisions (same coin from two strategies) are unambiguous.

Example — BTC from both strategies, ETH only from wf2, LINK only from bb:
```python
SELECTION = {
    'BTC_bb':  ('bb',  'BTC'),
    'BTC_wf2': ('wf2', 'BTC'),
    'ETH_wf2': ('wf2', 'ETH'),
    'LINK_bb': ('bb',  'LINK'),
}
```

In [ ]:
# ── sleeve selection ─────────────────────────────────────────────────────────
# label → (strategy_tag, coin)
SELECTION = {
    # wf_testing_2 sleeves
    'BTC_wf2':  ('wf2', 'BTC'),
    'ETH_wf2':  ('wf2', 'ETH'),
    'SOL_wf2':  ('wf2', 'SOL'),
    'XRP_wf2':  ('wf2', 'XRP'),

    # bb_breakout_wf sleeves
    'BTC_bb':   ('bb',  'BTC'),
    'ETH_bb':   ('bb',  'ETH'),
    'AVAX_bb':  ('bb',  'AVAX'),
    'LINK_bb':  ('bb',  'LINK'),
}

# ── build combined coin_dfs + symbol map ─────────────────────────────────────
coin_dfs     = {}   # label → oos_df
coin_symbols = {}   # label → binance spot coin (for hourly fetch)

missing = []
for label, (tag, coin) in SELECTION.items():
    if tag not in strategy_oos or coin not in strategy_oos[tag]:
        missing.append(f'{label} → {tag}:{coin}')
        continue
    coin_dfs[label]     = strategy_oos[tag][coin]
    coin_symbols[label] = coin

if missing:
    print('  [missing]')
    for m in missing:
        print(f'    {m}')

print(f'\nSelected {len(coin_dfs)} sleeves: {list(coin_dfs.keys())}')

---
## Hourly Data
Fetch hourly data for every **unique Binance symbol** referenced by the selected sleeves. Multiple sleeves on the same coin share one hourly pull.

In [ ]:
# ── fetch 1h data per unique coin (one-time pull) ────────────────────────────
hourly = {}   # label → 1h DataFrame (one entry per sleeve, sharing underlying symbol)
cache  = {}   # coin → hourly df (dedupe across sleeves)

for label, coin in coin_symbols.items():
    df = coin_dfs[label]
    if coin not in cache:
        symbol = coin + 'USDT'
        start  = str(df.index[0].date())
        end    = str(df.index[-1].date())
        klines = client.get_historical_klines(symbol, '1h', start, end)

        h = pd.DataFrame(klines, columns=[
            'Time','Open','High','Low','Close','Volume',
            'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
        ])
        h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
        for col in ['Open','High','Low','Close']:
            h[col] = h[col].astype(float)
        h = h.set_index('Time')
        cache[coin] = h
        print(f'  {coin}: {len(h)} hourly bars  ({start} → {end})')

    hourly[label] = cache[coin]

print('\nHourly data ready.')

---
### Scenarios
Re-aligns portfolio's exit/entry prices to a specific hour of the day. Signal generated on "Day T" is executed at the chosen hour on "Day T+1".

| Scenario | Entry | Exit |
|---|---|---|
| `'close'` | Close[T] midnight UTC — baseline (no modification) | Same |
| `integer` | HH:00 UTC on entry day T | HH:00 UTC on day after exit signal T'+1 |


In [ ]:
SCENARIO = 'close'   # ← 'close'  |  integer hour 0-23

def apply_scenario(coin_dfs, hourly, scenario):
    if scenario == 'close':
        return coin_dfs

    adjusted = {}
    for coin, df in coin_dfs.items():
        d = df.copy()
        h = hourly[coin].copy()

        # 1. Get the price at the specific hour
        prices_hh = h.loc[h.index.hour == scenario, 'Open'].copy()

        # 2. CRITICAL ALIGNMENT:
        # If scenario is 0 (00:00), this price is effectively the "Close"
        # of the PREVIOUS day. We must shift the index back by 1 day
        # so it aligns with the daily row that generated the signal.
        if isinstance(scenario, int):
            prices_hh.index = prices_hh.index.tz_convert(None) - pd.Timedelta(days=1)
            prices_hh.index = prices_hh.index.normalize()

        # 3. Map to the daily dataframe
        prices_hh = prices_hh[~prices_hh.index.duplicated(keep='first')]
        d['Close'] = prices_hh.reindex(d.index).ffill()

        adjusted[coin] = d

    return adjusted

coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h_UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'\nScenario applied: {label}')

---
## Portfolio Performance
Two-level weighting:

1. **`STRATEGY_WEIGHTS`** — how much of the book each strategy gets (e.g. `{'bb': 0.6, 'wf2': 0.4}`).
2. **`COIN_WEIGHTS`** — within each strategy, how that allocation splits across its sleeves. `None` for a strategy = equal-weight across its selected coins.

Final sleeve weight = `STRATEGY_WEIGHTS[strat] × coin_weight_within_strat`. Both dicts are auto-normalised, so values only need to be proportional.

| Parameter | Description |
|---|---|
| `show_coins` | Subset of sleeve labels to include. `None` = all |
| `benchmark` | `{'BTC': oos_df}` or multi-coin dict for B&H benchmark. `None` = equal-weight B&H of `show_coins` |

Per-sleeve statistics printed below, followed by a **cost stress test** (1×, 1.5×, 2×, 3× base cost).

In [ ]:
SHOW_COINS = list(coin_dfs.keys())   # or a manual subset, e.g. ['BTC_bb','ETH_wf2','SOL_wf2','LINK_bb']

# ── two-level weighting ──────────────────────────────────────────────────────
# 1) across strategies — None = equal-weight across strategies in SHOW_COINS
STRATEGY_WEIGHTS = None
# e.g. STRATEGY_WEIGHTS = {'bb': 0.6, 'wf2': 0.4}

# 2) within each strategy — None (or strat missing) = equal-weight across that
#    strategy's sleeves in SHOW_COINS
COIN_WEIGHTS = None
# e.g. COIN_WEIGHTS = {
#     'bb':  {'BTC': 0.4, 'ETH': 0.3, 'AVAX': 0.2, 'LINK': 0.1},
#     'wf2': {'BTC': 0.5, 'ETH': 0.3, 'SOL': 0.2},
# }

def _norm(d):
    s = sum(d.values())
    return {k: v / s for k, v in d.items()} if s > 0 else d

# build flat sleeve weights from the two levels
_strats_in_view = {SELECTION[s][0] for s in SHOW_COINS}
_sw = _norm(STRATEGY_WEIGHTS or {t: 1 for t in _strats_in_view})

sleeve_weights = {}
for tag in _strats_in_view:
    sleeves_of_tag = [s for s in SHOW_COINS if SELECTION[s][0] == tag]
    cw = (COIN_WEIGHTS or {}).get(tag)
    if cw is None:
        cw_norm = {SELECTION[s][1]: 1/len(sleeves_of_tag) for s in sleeves_of_tag}
    else:
        cw_norm = _norm({SELECTION[s][1]: cw.get(SELECTION[s][1], 0) for s in sleeves_of_tag})
    for s in sleeves_of_tag:
        sleeve_weights[s] = _sw[tag] * cw_norm[SELECTION[s][1]]

print('Sleeve weights (strategy × coin):')
for s in SHOW_COINS:
    tag, coin = SELECTION[s]
    print(f'  {s:<10}  strat={tag:<4} coin={coin:<5}  w={sleeve_weights[s]*100:>6.2f}%')
print(f'  {"":<10}  {"":<15}  {"total":<12} {sum(sleeve_weights.values())*100:>6.2f}%')

# pick any BTC sleeve as benchmark (original close series, not scenario-adjusted)
_btc_bench_label = next((l for l, (t, c) in SELECTION.items() if c == 'BTC'), None)
benchmark = {'BTC': coin_dfs[_btc_bench_label]} if _btc_bench_label else None

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
    weights    = sleeve_weights,
    show_coins = SHOW_COINS,
    benchmark  = benchmark,
    show       = True,
    cost       = 0.001,
    save_html  = None,
)


# ── per-sleeve trade statistics table ────────────────────────────────────────
print(f'\n── Per-Sleeve Trade Statistics  |  Scenario: {label} ──')
print(f'  {"Sleeve":<10} {"Strat":<6} {"Coin":<6} {"Trades":>7} {"Win Rate":>10} {"Prof.Factor":>13} {"Avg W/L":>9}')
print(f'  {"─"*10} {"─"*6} {"─"*6} {"─"*7} {"─"*10} {"─"*13} {"─"*9}')

for sleeve in SHOW_COINS:
    strat, coin = SELECTION[sleeve]
    t = metrics['coin_trade_stats'].get(sleeve, None)
    if t is None or len(t) == 0:
        print(f'  {sleeve:<10} {strat:<6} {coin:<6} {"0":>7} {"—":>10} {"—":>13} {"—":>9}')
        continue
    n      = len(t)
    wins   = t[t['pnl'] > 0]
    losses = t[t['pnl'] < 0]
    wr     = len(wins) / n if n else 0.0
    gp     = wins['pnl'].sum()
    gl     = abs(losses['pnl'].sum())
    pf     = gp / gl if gl > 0 else 0.0
    aw     = wins['pnl'].mean()        if len(wins)   > 0 else 0.0
    al     = abs(losses['pnl'].mean()) if len(losses) > 0 else 0.0
    awl    = aw / al                   if al          > 0 else 0.0
    print(f'  {sleeve:<10} {strat:<6} {coin:<6} {n:>7} {wr*100:>9.1f}% {pf:>13.2f} {awl:>9.2f}')

# ── portfolio cost stress test ───────────────────────────────────────────────
BASE_COST        = 0.001
cost_multipliers = (1.0, 1.5, 2.0, 3.0)

print(f"\n{'═'*72}")
print('PORTFOLIO TRANSACTION COST STRESS TEST')
print(f"{'═'*72}")
print(f'{"Cost":>8} {"Mult":>6} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Calmar":>8} {"PF":>8} {"WR":>8}')
print(f'{"─"*8} {"─"*6} {"─"*8} {"─"*10} {"─"*10} {"─"*8} {"─"*8} {"─"*8}')

for mult in cost_multipliers:
    c = BASE_COST * mult
    with contextlib.redirect_stdout(io.StringIO()):
        m = plot_portfolio_oos(
            coin_dfs   = coin_dfs_exec,
            weights    = sleeve_weights,
            show_coins = SHOW_COINS,
            benchmark  = benchmark,
            show       = False,
            cost       = c,
            save_html  = None,
        )
    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0
    print(f'{c:>8.4f} {mult:>5.1f}x {m["sharpe_ratio"]:>8.2f} {m["total_return"]*100:>9.2f}% '
          f'{m["max_drawdown"]*100:>9.2f}% {calmar:>8.2f} {m["profit_factor"]:>8.2f} {m["win_rate"]*100:>7.1f}%')